In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/jihyeseo/seoulairreport/SeoulHourlyAvgAirPollution.csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/jihyeseo/seoulairreport/SeoulHourlyAvgAirPollution.csv')

In [3]:
print(df.describe())

               측정일시  이산화질소농도(ppm)    오존농도(ppm)  일산화탄소농도(ppm)   아황산가스(ppm)  \
count  4.225000e+03   4187.000000  4183.000000   4183.000000  4187.000000   
mean   2.017112e+11      0.028922     0.013460      0.561009     0.004693   
std    2.017632e+04      0.015733     0.009577      0.236955     0.001304   
min    2.017112e+11      0.005000     0.001000      0.100000     0.002000   
25%    2.017112e+11      0.015000     0.004000      0.400000     0.004000   
50%    2.017112e+11      0.026000     0.012000      0.500000     0.005000   
75%    2.017112e+11      0.041000     0.021000      0.700000     0.006000   
max    2.017112e+11      0.092000     0.042000      1.800000     0.010000   

         미세먼지(㎍/㎥)   초미세먼지(㎍/㎥)  
count  4166.000000  4161.000000  
mean     36.724436    19.882480  
std      22.585665    12.798846  
min       5.000000     3.000000  
25%      22.000000    10.000000  
50%      30.000000    15.000000  
75%      46.000000    27.000000  
max     228.000000    85.000000  


In [4]:
df.rename(columns={'측정일시':'Date/Time','측정소명':'Location','이산화질소농도(ppm)':'NO2', '오존농도(ppm)':'O3','일산화탄소농도(ppm)':'CO','아황산가스(ppm)':'SO2','미세먼지(㎍/㎥)':'Fine Dust','초미세먼지(㎍/㎥)':'Ultrafine Dust'}, inplace=True)
df.head()

,Date/Time,Location,NO2,O3,CO,SO2,Fine Dust,Ultrafine Dust
0,201711242300,강남구,0.038,0.004,0.4,0.005,16.0,10.0
1,201711242200,강남구,0.031,0.008,0.4,0.005,17.0,9.0
2,201711242100,강남구,0.025,0.012,0.4,0.005,18.0,11.0
3,201711242000,강남구,0.033,0.007,0.4,0.005,21.0,12.0
4,201711241900,강남구,0.033,0.008,0.4,0.005,20.0,10.0


## 1강 1 : 결측치 해결

In [5]:
print(df.isnull().sum())

Date/Time          0
Location           0
NO2               38
O3                42
CO                42
SO2               38
Fine Dust         59
Ultrafine Dust    64
dtype: int64


In [6]:
altered = df.copy()
for col in altered.columns[2:]:
    altered[col] = altered[col].fillna(altered[col].median())

## 1장 2 : 이상치 제거

In [7]:
from scipy.stats import zscore

cols = ['NO2', 'O3', 'CO', 'SO2']
z = altered[cols].apply(zscore)

condition = (z.abs() <= 3).all(axis=1)
cleaned = altered[condition]

In [8]:
print(len(altered)-len(cleaned))

63
